<a href="https://colab.research.google.com/github/maithilikh/ExamPrepTool-RAG/blob/main/ExamPrepTool_RAG.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Pipeline:

```
PDFs → PyMuPDF extraction → OCR fallback (Tesseract)
→ Chunking (word-safe sliding window) → BGE-base embeddings → ChromaDB (persistent vector store)
→ Top-K semantic retrieval → Cross-encoder reranking → Top context selection → Prompt construction (mode-aware)
→ Groq Llama 3 streaming → Answer + citations + latency → Gradio UI (multi-mode)
```



---
Package installations and Setup

In [ ]:
!apt-get update -qq
!apt-get install -y tesseract-ocr

!pip -q install chromadb==1.1.0 sentence-transformers==5.1.0 groq==0.31.0 \
gradio==5.44.1 pymupdf==1.26.4 pytesseract pillow

In [ ]:
import os
import time
import hashlib
import tempfile

import fitz
import gradio as gr
import chromadb
import pytesseract
import numpy as np

from PIL import Image
from tqdm.auto import tqdm

from sentence_transformers import (SentenceTransformer, CrossEncoder)

from groq import Groq

In [ ]:
from google.colab import userdata

api_key = userdata.get("GROQ_API_KEY")
client = Groq(api_key=api_key)

MODEL_NAME = "llama-3.3-70b-versatile"

In [ ]:
MAX_FILES = 5
MAX_TOTAL_SIZE_MB = 100

CHUNK_SIZE = 700
CHUNK_OVERLAP = 150

TOP_K = 10
FINAL_K = 4

COLLECTION_NAME = "academic_docs"
DB_PATH = "./chroma_db"

In [ ]:
print("Loading embedding model...") # 440 mb
embedder = SentenceTransformer("BAAI/bge-base-en-v1.5")
print("Embedding model loaded")

In [ ]:
print("Loading reranker...")
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("Reranker loaded")

In [ ]:
db = chromadb.PersistentClient(path=DB_PATH) # Persistent db

collection = db.get_or_create_collection(name=COLLECTION_NAME)

In [ ]:
from pathlib import Path

def normalize_uploaded_files(files):
    """
    Converts Gradio upload objects into plain filesystem paths.
    Works across Gradio 4 and 5.
    """

    normalized = []
    for f in files:
        if isinstance(f, str):    # Already a filepath
            normalized.append(f)
        elif hasattr(f, "name"):    # NamedString / tempfile
            normalized.append(f.name)
        else:
            raise TypeError(f"Unsupported upload type: {type(f)}")
    return normalized

In [ ]:
def add_documents(ids, texts, vectors, metadata):
    collection.add(ids=ids, documents=texts,
        embeddings=vectors, metadatas=metadata
    )

def search_documents(query_vector, top_k):
    return collection.query(
        query_embeddings=[query_vector],
        n_results=top_k
    )

def document_count():
    return collection.count()

In [ ]:
def get_file_size_mb(path):
    return os.path.getsize(path) / (1024 * 1024)

In [ ]:
# def split_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
#     """Recursive splitter for chunking"""
#     chunks = []
#     start = 0
#     while start < len(text):
#         end = start + chunk_size
#         chunk = text[start:end]
#         chunks.append(chunk)
#         start += chunk_size - overlap
#     return chunks

def split_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Recursive splitter for chunking"""
    words = text.split()
    chunks = []
    current = []
    length = 0

    for word in words:
        current.append(word)
        length += len(word) + 1
        if length >= chunk_size:
            chunks.append(" ".join(current))
            overlap_words = current[-30:]
            current = overlap_words
            length = sum(len(w) + 1 for w in current)

    if current: chunks.append(" ".join(current))
    return chunks

In [ ]:
def embed(texts):
    return embedder.encode(
        texts,
        normalize_embeddings=True,
        show_progress_bar=False
    ).tolist()

In [ ]:
def reset_collection():
    """Clearing existing index"""
    global collection
    try:
        db.delete_collection(COLLECTION_NAME)
    except:
        pass

    collection = db.get_or_create_collection(name=COLLECTION_NAME)

In [ ]:
def file_hash(file):
    """(Future) To avoid re-indexing duplicate PDFs"""
    sha = hashlib.sha256()
    with open(file, "rb") as f:
        while True:
            data = f.read(8192)
            if not data:
                break
            sha.update(data)
    return sha.hexdigest()

---
PDF Processing, Vectore Indexing

In [ ]:
def run_ocr(page):
    pix = page.get_pixmap(dpi=200)
    img = Image.frombytes(
        "RGB",
        [pix.width, pix.height],
        pix.samples
    )
    text = pytesseract.image_to_string(img, config="--psm 6")
    return text.strip()

In [ ]:
def extract_pdf(file):
    """Tries normal text extraction, if very less text then OCR fallback"""
    doc = fitz.open(file)
    pages = []
    print("Extracting text...")

    for page_num in range(len(doc)):
        page = doc.load_page(page_num)
        text = page.get_text().strip()

        if len(text.strip()) < 200 and len(text.split()) < 30:
            text = run_ocr(page)

        pages.append({"page": page_num + 1, "text": text})
    doc.close()

    return pages

In [ ]:
def create_chunks(pages, file_name):
    """Chunking pages with file and page info"""
    docs = []
    print("Chunking", file_name, "...")
    for page in pages:
        chunks = split_text(page["text"])

        for i, chunk in enumerate(chunks):
            if len(chunk.strip()) < 30:
                continue
            docs.append({
                "text": chunk, "page": page["page"], "chunk": i,
                "file": file_name
            })
    return docs

In [ ]:
# def add_to_collection(docs):
#     """Adds chunks to Chroma"""
#     texts = [d["text"] for d in docs]
#     vectors = embed(texts)
#     ids = []
#     metadata = []

#     for d in docs:
#         ids.append(f'{d["file"]}_{d["page"]}_{d["chunk"]}')
#         metadata.append({
#             "file": d["file"],
#             "page": d["page"]
#         })

#     #collection.add(ids=ids, documents=texts, embeddings=vectors, metadatas=metadata)
#     add_documents(ids, texts, vectors, metadata)

def add_to_collection(docs):
    """Adds chunks to Chroma"""
    batch_size = 64
    for i in range(0, len(docs), batch_size):
        batch = docs[i:i+batch_size]
        texts = [d["text"] for d in batch]
        vectors = embed(texts)
        ids = [
            f'{d["file"]}_{d["page"]}_{d["chunk"]}'
            for d in batch
        ]
        metadata = [
            {"file": d["file"], "page": d["page"]}
            for d in batch
        ]
        add_documents(ids, texts, vectors, metadata)

In [ ]:
def validate_files(files):
    if len(files) == 0:
        raise ValueError("Upload at least one PDF.")

    if len(files) > MAX_FILES:
        raise ValueError(f"Maximum {MAX_FILES} PDFs allowed.")

    total = 0
    print(type(files[0]))
    for file in files:
        total += get_file_size_mb(file)

    if total > MAX_TOTAL_SIZE_MB:
        raise ValueError(f"Combined PDF size exceeds {MAX_TOTAL_SIZE_MB} MB.")

In [ ]:
def total_pages(files):
    """Counts pages"""
    pages = 0
    for file in files:
        doc = fitz.open(file)
        pages += len(doc)
        doc.close()
    return pages

In [ ]:
def build_index(files):
    print("Starting indexing...")
    validate_files(files)
    print("Validation done.")
    pages = total_pages(files)
    print("Pages count:", pages)

    if pages > 1200:
        raise ValueError("Too many pages uploaded.")

    reset_collection()
    print("Collection reset")
    total_chunks = 0
    start = time.time()

    for file in tqdm(files):
        pdf_pages = extract_pdf(file)
        docs = create_chunks(
            pdf_pages,
            os.path.basename(file)
        )
        add_to_collection(docs)
        total_chunks += len(docs)
        print(f"Finished {os.path.basename(file)}")

    elapsed = round(time.time() - start, 2)
    print(f"Finished indexing in {elapsed}s")

    return {
        "chunks": total_chunks,
        "pages": pages,
        "time": elapsed
    }

In [ ]:
def collection_stats():
    """Database info"""
    # count = collection.count()
    document_count()
    print(f"Indexed chunks : {count}")

In [ ]:
# Indexing test
# info = build_index(files)
# print(info)
# collection_stats()

---
Retrieval, Reranking and Groq

In [ ]:
def retrieve(query, top_k=TOP_K):
    """Retrieves relevant chunks"""
    query_embedding = embed([query])[0]
    # results = collection.query(
    #     query_embeddings=[query_embedding],
    #     n_results=top_k
    # )
    results = search_documents(query_embedding, top_k)
    docs = results["documents"][0]
    meta = results["metadatas"][0]
    print(f"Retrieved {len(docs)} chunks")
    return docs, meta

In [ ]:
def rerank(query, docs, meta):
    """Reranks the results"""
    pairs = [[query, doc] for doc in docs]
    scores = reranker.predict(pairs)
    ranked = list(zip(scores, docs, meta))
    ranked.sort(key=lambda x: x[0], reverse=True)
    ranked = ranked[:FINAL_K]
    final_docs = [item[1] for item in ranked]
    final_meta = [item[2] for item in ranked]
    return final_docs, final_meta

In [ ]:
# def build_prompt(question, docs):
#     context = "\n\n".join(docs)
#     prompt = f""" You are an academic assistant.
#       Use only the information provided in the context.

#       If the answer is missing from the documents, say you couldn't find enough information.
#       Do not make up facts.

#       Explain clearly in simple language.

#       Context:
#       {context}

#       Question:
#       {question}

#       Answer:
#     """
#     return prompt

def build_prompt(question, docs, mode):
    context = "\n\n".join(docs)
    if mode == "Ask":
        return f"""
        You are an academic assistant.

        Use only the given context.
        If answer is missing, say you couldn't find it.

        Context:
        {context}

        Question:
        {question}

        Answer:
        """

    if mode == "Summarize":
        return f"""
        You are an academic assistant.
        Summarize the content clearly in simple points.

        Context:
        {context}

        Task:
        {question}

        Summary:
        """

    if mode == "Quiz Me":

        return f"""
          You are an academic assistant.

          Generate 5-10 exam questions from the context.
          Also provide short answers.

          Context:
          {context}

          Topic:
          {question}

          Output format:
          Q1...
          A1...
          """

In [ ]:
def format_sources(meta):
    seen = set()
    lines = []
    for item in meta:
        source = (
            item["file"],
            item["page"]
        )
        if source not in seen:
            seen.add(source)
            lines.append(
                f'{item["file"]} (Page {item["page"]})'
            )
    return "\n".join(lines)
    # return "\n".join( f"• {line.replace('(Page', '- Page').replace(')', '')}" for line in lines)

In [ ]:
history = []
MAX_HISTORY = 6

In [ ]:
def stream_answer(question, mode):
    """Streaming Response"""
    docs, meta = retrieve(question)
    docs, meta = rerank(question, docs, meta)

    prompt = build_prompt(question, docs, mode)
    messages = []

    for user, assistant in history[-MAX_HISTORY:]:
        messages.append({
            "role": "user",
            "content": user
        })
        messages.append({
            "role": "assistant",
            "content": assistant
        })

    messages.append({
        "role": "user",
        "content": prompt
    })

    stream = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        stream=True,
        temperature=0.2
    )
    answer = ""

    for chunk in stream:
        delta = chunk.choices[0].delta.content
        if delta:
            answer += delta
            yield answer, meta

    history.append( (question, answer) )

    if len(history) > MAX_HISTORY: history.pop(0)

In [ ]:
def ask(question, mode):
    """Question Answer function (for Gradio)"""
    start = time.time()
    final_answer = ""
    sources = ""
    for partial, meta in stream_answer(question, mode):
        final_answer = partial
        sources = format_sources(meta)
        yield (
            final_answer,
            sources,
            f"{time.time()-start:.2f} sec"
        )

In [ ]:
def clear_history():
    """Reset Conversation"""
    history.clear()

In [ ]:
# Quick console test
# For answer, sources, latency in ask(
#     "Explain deadlock."
# ):
#     pass
#
# print(answer)
# print(sources)
# print(latency)

---
Gradio UI

In [ ]:
index_ready = False

def process_documents(files):
    """Process uploaded PDFs"""
    global index_ready
    if not files:
        return ( "Upload at least one PDF.", gr.update(interactive=False))
    files = normalize_uploaded_files(files)
    info = build_index(files)
    index_ready = True
    summary = (
        f"Indexed {len(files)} PDF(s)\n"
        f"Pages : {info['pages']}\n"
        f"Chunks : {info['chunks']}\n"
        f"Time : {info['time']} sec"
    )
    return (
        summary,
        gr.update(interactive=True),
        gr.update(interactive=True)
    )

In [ ]:
def chat(question, mode):
    # Chat wrapper
    if not index_ready:
        yield ("Please process documents first.", "", "")
        return

    for answer, sources, latency in ask(question, mode):
        yield (answer, sources, latency)

In [ ]:
def reset():
    global index_ready
    clear_history()
    reset_collection()
    index_ready = False
    return (
        "", "", "", "",
        gr.update(interactive=False),
        gr.update(value="Ask")
    )

In [ ]:
with gr.Blocks(title="Academic Document Assistant", theme=gr.themes.Ocean()) as demo:
    gr.Markdown("# Academic Document Assistant")
    gr.Markdown(
        "Upload one or more PDFs, process them once, then ask questions."
    )

    with gr.Row():
        with gr.Column(scale=1):
            files = gr.File(
                label="Upload PDFs",
                file_count="multiple",
                file_types=[".pdf"]
            )
            process_btn = gr.Button("Process Documents", variant="primary")
            reset_btn = gr.Button("Reset")
            status = gr.Textbox( label="Status", interactive=False, lines=5)

        with gr.Column(scale=2):
            mode = gr.Radio(
                choices=["Ask", "Summarize", "Quiz Me"],
                value="Ask",
                label="Mode"
            )
            question = gr.Textbox(
                label="Question",
                placeholder="Ask something about the uploaded documents...",
                interactive=False
            )
            ask_btn = gr.Button("Ask")
            answer = gr.Textbox(label="Answer", lines=12)

            with gr.Row():
                sources = gr.Textbox(label="Sources", lines=5)
                latency = gr.Textbox(label="Response Time")

    process_btn.click(
        process_documents,
        inputs=files,
        outputs=[status, question, mode]
    )

    ask_btn.click(
        chat,
        inputs=[question, mode],
        outputs=[answer, sources, latency]
    )

    reset_btn.click(
        reset,
        outputs=[status, answer, sources, latency, question, mode]
    )

In [ ]:
demo.queue()

demo.launch(debug=True)